In [ ]:
import pandas as pd
import json
from aloud_database.aloud_database import Database

query = """
SELECT
    *
FROM
    lead_tracking_prod.conversions c 
WHERE
    c.campaign_id = 'isca-aulassemanais'
    and c.conversion_type_id = '2'
"""

db = Database()

df = db.execute_query(query=query)
# Supondo que seu DataFrame seja df e a coluna seja 'conversion_raw_info'

#df = df[df['campaign_id'] == 'BF24']

# Se a coluna vier como string JSON, converter para dict
df["conversion_raw_info"] = df["conversion_raw_info"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

# Função recursiva para extrair todas as chaves de um JSON
def extract_keys(obj, parent_key=""):
    keys = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            full_key = f"{parent_key}.{k}" if parent_key else k
            keys.add(full_key)
            keys |= extract_keys(v, full_key)
    elif isinstance(obj, list):
        for i, item in enumerate(obj):
            keys |= extract_keys(item, parent_key)
    return keys

# Aplicar em todas as linhas
all_keys = set()
for row in df["conversion_raw_info"].dropna():
    all_keys |= extract_keys(row)

# Converter para lista ordenada
all_keys_list = sorted(all_keys)

# Exibir resultado
for k in all_keys_list:
    print(k)

In [ ]:
# Lista de variações possíveis
income_keys = [
    "monthly_incomme",
    "monthy_income"
]

teb = [
    "took_english_course",
    "took_english_course_before"
]

# Função para extrair o valor de uma lista de keys do JSON
def get_value_from_keys(row, keys):
    if not isinstance(row, dict):
        return None
    for k in keys:
        if k in row:
            return row[k]
    return None

# Criando as novas colunas usando a função adaptada
df["monthly_incomme"] = df["conversion_raw_info"].apply(lambda x: get_value_from_keys(x, income_keys))

# Criando as novas colunas usando a função adaptada
df["teb"] = df["conversion_raw_info"].apply(lambda x: get_value_from_keys(x, teb))

df["age_range"] = df["conversion_raw_info"].apply(
    lambda x: x.get("age_range") if isinstance(x, dict) else None
)

df["gender"] = df["conversion_raw_info"].apply(
    lambda x: x.get("gender") if isinstance(x, dict) else None
)

df["current_occupation"] = df["conversion_raw_info"].apply(
    lambda x: x.get("current_occupation") if isinstance(x, dict) else None
)